# Этап 3. Сборка таблицы «день × район»

**Цель:** превратить 16 млн отдельных вызовов в одну аккуратную таблицу, на которой будут учиться все модели (этапы 4–7).

Одна строка = один день в одном районе. Колонки:
- **что предсказываем:** `n_calls` — число вызовов;
- **для H2:** `resp_median` — медианное время доезда;
- **погода:** температура, влажность, осадки;
- **календарь:** день недели, месяц, праздник, перевод часов;
- **флаг COVID**, чтобы потом исключить этот период из обучения.

Правила те же: `# ТВОЙ КОД` пишешь ты, после каждого шага **Вывод:** своими словами.

In [1]:
import duckdb
import pandas as pd

EMS = "../data/raw/ems/*.parquet"
WEATHER = "../data/raw/weather_daily.parquet"
OUT = "../data/processed/daily_borough.parquet"

con = duckdb.connect()

## 1. Что выбрасываем

В EDA мы нашли «мусор»:
- `STNDBY` — это не вызов от человека, а дежурство бригады (например, на пожаре или мероприятии). Бывают дни, когда их сотни, и это шум, не связанный с жарой.
- `borough = 'UNKNOWN'` — вызов без района, его не к чему привязать.

**Задание:** одним SQL-запросом посчитай, сколько строк попадает под каждое условие и какую это составляет долю от всех 16,5 млн.

Подсказка: внутри `SELECT` можно писать `sum(CASE WHEN условие THEN 1 ELSE 0 END)`, или короче `count(*) FILTER (WHERE условие)`.

In [2]:
con.sql(f"""
SELECT
    count (*)                                        AS total,
    count(*) FILTER (WHERE initial_call_type = 'STNDBY') AS n_stndby,
    count(*) FILTER (WHERE borough = 'UNKNOWN')      AS n_unknown,
    n_stndby/total * 100 AS pct_stndby,
    n_unknown/total * 100 AS pct_unknown
FROM '{EMS}'
""").df()

,total,n_stndby,n_unknown,pct_stndby,pct_unknown
0,16521627,43154,64,0.261197,0.000387


**Вывод:** STNDBY — это не вызов от пациента, а дежурство бригады. Мы изучаем, как жара влияет на людей, которые звонят в скорую, поэтому STNDBY не относится к нашему вопросу. Их доля всего 0,26 %. Для вызовов без района невозможно определить погоду, их ещё меньше (0,0004 %). Удаление безопасно.

## 2. Главная агрегация: SQL по всем 16 млн строк

Напиши запрос, который **без** `STNDBY` и `UNKNOWN` считает для каждой пары (дата, район):
- `n_calls` — все вызовы;
- `n_severe` — тяжёлые вызовы, `initial_severity_level_code <= 3` (пригодится позже, жара бьёт по ним сильнее);
- `resp_median` — медиану `incident_response_seconds_qy`, но **только** там, где время доезда считается корректным.

Сначала выясни, какие значения бывают в `valid_incident_rspns_time_indc` (маленький запрос с `GROUP BY`). Медиана в DuckDB: `median(x)`. Медиана только по части строк: `median(x) FILTER (WHERE ...)`.

Почему медиана, а не среднее? Подумай, что сделают с средним 10 вызовов, где бригада ехала 3 часа.

Результат сохрани в `calls`.

In [3]:
con.sql(f"""
SELECT count(*) AS count,
valid_incident_rspns_time_indc 
FROM '{EMS}'
GROUP BY valid_incident_rspns_time_indc """).df()

,count,valid_incident_rspns_time_indc
0,693844,N
1,15827783,Y


In [4]:
calls = con.sql(f""" 
SELECT count(*) AS n_calls,
       count(*) FILTER (WHERE initial_severity_level_code <= 3) AS n_severe,
       median(incident_response_seconds_qy) FILTER(WHERE valid_incident_rspns_time_indc = 'Y')  AS resp_median,
       CAST(incident_datetime AS DATE) AS date,
       borough 
FROM '{EMS}' 
WHERE initial_call_type != 'STNDBY' AND borough != 'UNKNOWN'
GROUP BY date, borough
ORDER BY date, borough; """).df()  

len(calls) 


19630

## 3. Проверка полноты

Дней с 2016-01-01 по 2026-09-30 — 3 926, районов — 5. Значит, строк должно быть ровно **19 630**.

**Задание:**
1. Проверь `len(calls)`.
2. Если меньше, найди, каких пар не хватает. Идея: построй «полную сетку» всех дат × всех районов (`pd.date_range` + `pd.MultiIndex.from_product`, или `merge(how="cross")`) и сравни с `calls`.
3. Проверь, нет ли дней с подозрительно малым числом вызовов (например, `n_calls` меньше половины медианы для своего района). Это могут быть дыры в выгрузке.

**Вопрос:** если для пары нет ни одного вызова, это «0 вызовов» или «нет данных»? Как поступим?

In [5]:
calls["med"] = calls.groupby("borough")["n_calls"].transform("median")

In [6]:
con.sql(f""" 
SELECT date,
       n_calls,
       borough,
       med
FROM calls
WHERE n_calls < 0.7 * med; """).df()

,date,n_calls,borough,med
0,2016-02-21,121,RICHMOND / STATEN ISLAND,177.0
1,2016-02-28,121,RICHMOND / STATEN ISLAND,177.0
2,2016-11-24,118,RICHMOND / STATEN ISLAND,177.0
3,2017-03-14,706,MANHATTAN,1038.0
4,2018-02-25,123,RICHMOND / STATEN ISLAND,177.0
5,2020-04-18,722,MANHATTAN,1038.0
6,2020-04-19,719,MANHATTAN,1038.0
7,2020-04-23,681,MANHATTAN,1038.0
8,2020-04-24,122,RICHMOND / STATEN ISLAND,177.0
9,2020-04-26,681,MANHATTAN,1038.0


**Вывод:** Все 19 630 пар «день × район» на месте, пропусков нет. Дней с вызовами меньше 50 % от медианы района нет, значит, дыр в выгрузке нет. При пороге 70 % нашлось 23 дня, и все объясняются реальными событиями:
- весна 2020: локдаун, в основном Манхэттен;
- День благодарения (24.11.2016, 26.11.2020);
- метель 14.03.2017: люди сидели дома, а вызвать скорую в снегопад тоже сложнее;
- воскресенья в Стейтен-Айленде: район маленький (около 177 вызовов в день), поэтому случайные колебания в процентах там сильнее.

In [7]:
calls = calls.drop(columns="med")


## 4. Присоединяем погоду

У погоды и у вызовов районы должны называться одинаково. Сравни `calls["borough"].unique()` и `weather["borough"].unique()`.

Потом соедини по **двум** ключам: `on=["date", "borough"]`. Проверь, что после `merge` число строк не изменилось и в колонках погоды нет пропусков.

Подсказка: если типы `date` разные (у одной таблицы `date`, у другой `datetime`), приведи обе к `pd.to_datetime`.

In [8]:
weather = pd.read_parquet(WEATHER)

In [9]:
df = calls.merge(weather, on = ["date", "borough"])

In [10]:
len(df)

19630

In [11]:
df.isna().sum()

n_calls                      0
n_severe                     0
resp_median                  0
date                         0
borough                      0
temperature_2m_max           0
temperature_2m_min           0
temperature_2m_mean          0
apparent_temperature_max     0
relative_humidity_2m_mean    0
precipitation_sum            0
dtype: int64

**Вывод:** соединили погоду и вызовы по дате и районам, число строк не изменилось, пропусков нет

## 5. Календарные признаки

Добавь колонки:
- `weekday` — номер дня недели 0–6 (`.dt.dayofweek`); числа модели удобнее, чем названия;
- `month`, `year`;
- `is_holiday` — федеральный праздник США:
  ```python
  from pandas.tseries.holiday import USFederalHolidayCalendar
  holidays = USFederalHolidayCalendar().holidays(start="2016-01-01", end="2026-12-31")
  ```
  затем `df["date"].isin(holidays)`;
- `is_dst_change` — день перевода часов. В США это 2-е воскресенье марта и 1-е воскресенье ноября. Попробуй вычислить сама: воскресенье, месяц 3, и число от 8 до 14. Для ноября — число от 1 до 7.

**Проверка:** найди в таблице 27.11.2025 (День благодарения) и 09.03.2025. Флаги должны стоять.

In [12]:
from pandas.tseries.holiday import USFederalHolidayCalendar
holidays = USFederalHolidayCalendar().holidays(start="2016-01-01", end="2026-12-31")

In [13]:
df["weekday"] = df["date"].dt.dayofweek


In [14]:
df["month"] = df["date"].dt.month


In [15]:
df["year"] = df["date"].dt.year

In [16]:
df["is_holiday"] = df["date"].isin(holidays)

In [17]:
df["is_dst_change"] = (
    ((df["date"].dt.month == 3) &
     (df["date"].dt.day.between(8, 14)) &
     (df["date"].dt.dayofweek == 6))
    |
    ((df["date"].dt.month == 11) &
     (df["date"].dt.day.between(1, 7)) &
     (df["date"].dt.dayofweek == 6))
)

## 6. Флаг COVID

В EDA ты решила, что ненормальный период нельзя давать модели. Нужна граница.

Добавь `is_covid`: `True` с **2020-03-01** по **2022-02-28** (от первого локдауна до спада Омикрона).

**Вопрос, ответь своими словами:** почему мы ставим флаг, а не удаляем строки прямо сейчас? (Подсказка: этап 4 и график для README.)

In [18]:
df["date"] = pd.to_datetime(df["date"])
df["is_covid"] = df["date"].between("2020-03-01", "2022-02-28")

In [19]:
df["is_covid"].sum() 

np.int64(3650)

**Ответ:** для наглядности оставим COVID в данных с флагом, чтобы показать, как сломалась система в этот период; из моделей его исключим

## 7. Финальная проверка и сохранение

1. `df.info()` — типы, пропуски.
2. `df.describe()` — нет ли отрицательных вызовов, температуры +80 и т.п.
3. Отсортируй по `["borough", "date"]` и сохрани: `df.to_parquet(OUT, index=False)`.
4. Быстрый санити-чек: средний `n_calls` по районам. Сходится ли с тем, что Бруклин самый загруженный, а Стейтен-Айленд самый маленький?

In [20]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 19630 entries, 0 to 19629
Data columns (total 17 columns):
 #   Column                     Non-Null Count  Dtype         
---  ------                     --------------  -----         
 0   n_calls                    19630 non-null  int64         
 1   n_severe                   19630 non-null  int64         
 2   resp_median                19630 non-null  float64       
 3   date                       19630 non-null  datetime64[us]
 4   borough                    19630 non-null  str           
 5   temperature_2m_max         19630 non-null  float64       
 6   temperature_2m_min         19630 non-null  float64       
 7   temperature_2m_mean        19630 non-null  float64       
 8   apparent_temperature_max   19630 non-null  float64       
 9   relative_humidity_2m_mean  19630 non-null  int64         
 10  precipitation_sum          19630 non-null  float64       
 11  weekday                    19630 non-null  int32         
 12  month          

In [21]:
df.describe()

,n_calls,n_severe,resp_median,date,temperature_2m_max,temperature_2m_min,temperature_2m_mean,apparent_temperature_max,relative_humidity_2m_mean,precipitation_sum,weekday,month,year
count,19630.000000,19630.000000,19630.000000,19630,19630.000000,19630.000000,19630.000000,19630.000000,19630.000000,19630.000000,19630.000000,19630.000000,19630.000000
mean,839.451350,303.880897,481.124707,2021-05-16 12:00:00,16.993597,8.605909,12.687677,15.500764,70.198625,3.540993,3.000000,6.417728,2020.882068
min,118.000000,35.000000,315.000000,2016-01-01 00:00:00,-12.900000,-22.100000,-15.000000,-20.600000,30.000000,0.000000,0.000000,1.000000,2016.000000
25%,766.000000,270.250000,416.000000,2018-09-08 00:00:00,8.900000,0.900000,4.900000,5.100000,60.000000,0.000000,1.000000,3.000000,2018.000000
50%,951.000000,340.000000,458.000000,2021-05-16 12:00:00,17.400000,8.600000,12.900000,15.600000,71.000000,0.100000,3.000000,6.000000,2021.000000
75%,1084.000000,393.000000,521.000000,2024-01-23 00:00:00,25.600000,17.100000,21.300000,26.600000,81.000000,3.100000,5.000000,9.000000,2024.000000
max,1840.000000,761.000000,1417.000000,2026-09-30 00:00:00,41.100000,29.200000,34.800000,44.500000,100.000000,156.700000,6.000000,12.000000,2026.000000
std,358.537593,130.097760,94.522002,NaN,9.857645,9.426740,9.452067,12.584261,13.414525,8.067563,2.000306,3.417139,3.104248


In [22]:
df = df.sort_values(["borough", "date"])

In [23]:
df.to_parquet(OUT, index=False)

In [24]:
df.groupby("borough")["n_calls"].mean().sort_values(ascending=False)

borough
BROOKLYN                    1161.134233
MANHATTAN                   1034.545593
BRONX                        987.426898
QUEENS                       836.391493
RICHMOND / STATEN ISLAND     177.758533
Name: n_calls, dtype: float64

In [25]:
df.nlargest(3, "resp_median")[["date", "borough", "resp_median"]]

,date,borough,resp_median
18530,2026-02-23,BRONX,1417.0
18534,2026-02-23,RICHMOND / STATEN ISLAND,1219.5
18535,2026-02-24,BRONX,1133.5


Это дни сильной метели (23–24.02.2026): мощный снегопад, нерасчищенные дороги, вероятно, много ДТП и травм.

In [26]:
df.nlargest(3, "n_calls")[["date", "borough", "n_calls"]]

,date,borough,n_calls
7751,2020-03-30,BROOKLYN,1840
7761,2020-04-01,BROOKLYN,1797
7736,2020-03-27,BROOKLYN,1784


На эти даты приходится пик COVID в Нью-Йорке.

In [27]:
df.nlargest(3, "precipitation_sum")[["date", "borough", "precipitation_sum"]]

,date,borough,precipitation_sum
12342,2022-10-04,MANHATTAN,156.7
10352,2021-09-01,MANHATTAN,129.3
10350,2021-09-01,BRONX,128.2


Осадки в данных — это оценка реанализа ERA5, а не замер метеостанции. Например, 04.10.2022 модель даёт 100–157 мм по всем районам, хотя сильного ливня в новостях нет. Для анализа используем осадки только как вспомогательный признак.

## Итоги этапа 3

1. Сколько строк и колонок в итоговой таблице, сколько дней «чистых» (без COVID)?
2. Что выбросили и почему?
3. Какие проблемы с данными нашла на шаге 3?

После этого: коммит `Этап 3: таблица день × район`, и переходим к этапу 4 (регрессия Пуассона, проверка H1).

1. **Итоговая таблица:** 19 630 строк × 17 колонок (3 926 дней × 5 районов). Без COVID 15 980 строк, то есть 3 196 «чистых» дней. Сохранена в `data/processed/daily_borough.parquet`.
2. **Что выбросили:** около 43 тыс. вызовов (0,26 %). Это `STNDBY`, дежурства бригад, а не вызовы пациентов, и вызовы с районом `UNKNOWN`, которые нельзя привязать к погоде. Период COVID не удаляли, а пометили флагом `is_covid`, чтобы показать его на графиках и решить, что исключать, уже при обучении.
3. **Проблемы с данными:**
   - пропусков нет, все пары «день × район» на месте; самые низкие дни объясняются реальными событиями (локдаун, праздники, метель 14.03.2017);
   - время доезда зависит не только от нагрузки: в снегопад 23–24.02.2026 медиана выросла до 24 минут. Для H2 надо учитывать погоду и дороги;
   - максимум вызовов (1 840 в Бруклине, 30.03.2020) — пик COVID, ещё один довод за флаг;
   - осадки из реанализа ERA5 ненадёжны: 04.10.2022 модель даёт 100–157 мм, а сильного ливня не было. Используем их только как вспомогательный признак.